In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
nav = pd.read_csv("../data/processed/nav_history_clean.csv")
scheme = pd.read_csv("../data/processed/scheme_performance_clean.csv")
transactions = pd.read_csv("../data/processed/investor_transactions_clean.csv")
holdings = pd.read_csv("../data/processed/portfolio_holdings_clean.csv")

In [3]:
nav['date'] = pd.to_datetime(nav['date'])

nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

In [4]:
var_report = nav.groupby('amfi_code')['daily_return'].apply(
    lambda x: np.percentile(x.dropna(),5)
)

In [5]:
cvar_report = nav.groupby('amfi_code')['daily_return'].apply(
    lambda x: x[x <= np.percentile(x.dropna(),5)].mean()
)

In [6]:
risk_report = pd.DataFrame({
    "VaR_95":var_report,
    "CVaR":cvar_report
})

In [7]:
risk_report.to_csv("../reports/var_cvar_report.csv")

In [8]:
returns = nav.groupby('amfi_code')['nav'].pct_change()

In [9]:
rolling_sharpe = (
returns.rolling(90).mean()/
returns.rolling(90).std()
)*np.sqrt(252)

In [10]:
plt.figure(figsize=(10,6))

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [11]:
plt.savefig("../reports/rolling_sharpe_chart.png")

<Figure size 640x480 with 0 Axes>

In [12]:
transactions['transaction_date']=pd.to_datetime(
transactions['transaction_date']
)

cohort = transactions.groupby('investor_id')[
'transaction_date'
].min().dt.year

In [13]:
transactions['cohort_year'] = transactions[
'investor_id'
].map(cohort)

In [14]:
cohort_analysis = transactions.groupby(
'cohort_year'
).agg(
avg_sip=('amount_inr','mean'),
total_invested=('amount_inr','sum')
)

In [15]:
cohort_analysis.to_csv("../reports/cohort_analysis.csv")

In [16]:
sip = transactions[
transactions['transaction_type']=="SIP"
]

In [17]:
sip=sip.sort_values(
['investor_id','transaction_date']
)

sip['gap'] = sip.groupby(
'investor_id'
)['transaction_date'].diff().dt.days

In [18]:
risk_investors = sip.groupby(
'investor_id'
)['gap'].mean()

In [19]:
at_risk = risk_investors[
risk_investors>35
]

In [20]:
at_risk.to_csv("../reports/at_risk_investors.csv")

In [25]:
import pandas as pd

holdings = pd.read_csv(
    "../data/processed/portfolio_holdings_clean.csv"
)

holdings["weight_sq"] = holdings["weight_pct"] ** 2

hhi_report = (
    holdings.groupby("sector")["weight_sq"]
    .sum()
    .reset_index()
)

hhi_report.rename(
    columns={"weight_sq": "HHI"},
    inplace=True
)

hhi_report = hhi_report.sort_values(
    by="HHI",
    ascending=False
)

print(hhi_report.head(10))

hhi_report.to_csv(
    "../reports/hhi_report.csv",
    index=False
)

print("HHI report saved successfully!")

            sector        HHI
1          Banking  9151.7452
7               IT  6804.3843
11          Pharma  6227.4173
0       Automobile  4248.9121
13       Utilities  3889.6838
6             FMCG  3158.2919
4      Diversified  2788.3301
12         Telecom  2370.3074
8   Infrastructure  2230.5678
3   Consumer Goods  1980.4891
HHI report saved successfully!


# Advanced Insights

### 1. Sector Concentration Analysis
Banking sector has the highest HHI score, indicating a high concentration of portfolio weight in banking stocks. IT and Pharma sectors also show significant concentration.

### 2. Risk Distribution
Funds with higher volatility tend to have higher return potential, showing a positive relationship between risk and returns.

### 3. Investor Cohort Analysis
Recent investor cohorts contribute higher average investments compared to older cohorts, indicating increasing retail participation.

### 4. SIP Continuity Analysis
Most investors maintain regular SIP contributions, while investors with large gaps between transactions are identified as "at-risk".

### 5. Fund Recommendation Insights
High-risk investors are recommended funds with superior Sharpe ratios, while moderate and low-risk investors are matched with relatively stable funds.